# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.



In [2]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print(os.getcwd())

/content/Flyrank-ML-Internship


## 1. Distributions

Before testing any signals, I inspected the distributions of the main
performance fields. Several metrics show a heavy right tail: most pages
have relatively small values, while a small number of pages receive very
high impressions, clicks, or sessions.

This matters because averages can be misleading in skewed data. For
heavily skewed fields, medians and percentile comparisons provide a
more stable view of typical behavior.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same filters used in previous notebooks
df = df[(df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)].copy()

key_fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate"
]

print(df[key_fields].describe().T)

for col in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    print(f"\n{col}")
    print(df[col].quantile([0.50, 0.75, 0.90, 0.95, 0.99]))

                   count         mean           std  min   25%     50%  \
impressions_90d  30000.0  5200.366300  16838.019547  1.0  81.0  731.00   
clicks_90d       30000.0    16.097333     75.076958  0.0   0.0    1.00   
sessions_90d     30000.0    37.066633    107.069131  1.0   2.0    7.00   
ctr              30000.0     0.510733      3.279162  0.0   0.0    0.07   
avg_position     30000.0    16.342380     15.216790  0.0   6.2   10.80   
engagement_rate  30000.0     2.534520      8.310096  0.0   0.0    0.00   

                     75%       max  
impressions_90d  3615.25  517715.0  
clicks_90d          7.00    4178.0  
sessions_90d       27.00    4345.0  
ctr                 0.29     100.0  
avg_position       22.30     245.0  
engagement_rate     1.35     100.0  

impressions_90d
0.50      731.00
0.75     3615.25
0.90    12136.40
0.95    22996.50
0.99    73505.83
Name: impressions_90d, dtype: float64

clicks_90d
0.50      1.00
0.75      7.00
0.90     32.00
0.95     69.05
0.99    25

## 2. Signal #1: Better position → higher CTR

Signal #1

Hypothesis:
Pages with better search positions should generally receive higher CTR.

Verdict:
[Fill after running code: CONFIRMED / MIXED / OPPOSITE / FALSE]

In [4]:
position_corr = df["avg_position"].corr(df["ctr"])

print("Correlation between average position and CTR:")
print(position_corr)

Correlation between average position and CTR:
-0.07259029976832464


## Signal #2: More impressions → more clicks

Signal #2

Hypothesis:
Pages that receive more impressions should generally receive more clicks.

Verdict:
[Fill after running code]

In [5]:
impression_click_corr = df["impressions_90d"].corr(df["clicks_90d"])

print("Correlation between impressions and clicks:")
print(impression_click_corr)

Correlation between impressions and clicks:
0.6962810028467799


## Signal #3: Higher engagement → less decline

Signal #3

Hypothesis:
Pages with stronger engagement should be less likely to appear in the
declining group.

Verdict:
[Fill after running code]

In [6]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

engagement_by_label = (
    df.groupby("is_declining")["engagement_rate"]
      .mean()
)

print(engagement_by_label)

is_declining
0    2.649728
1    2.437193
Name: engagement_rate, dtype: float64


## 3. The flag-linked test

Flag-linked test

One assumption behind refresh prioritization is that pages classified as
declining show weaker search performance than non-declining pages.

I tested whether declining pages have lower CTR and weaker engagement
than other pages.

In [7]:
flag_test = (
    df.groupby("trend_direction")
      [["ctr", "engagement_rate", "avg_position"]]
      .mean()
)

print(flag_test)

                      ctr  engagement_rate  avg_position
trend_direction                                         
down             0.324138         2.437193     15.936305
flat             1.376753         1.374514     11.102431
new              1.296521         2.135653      9.912433
stable           0.517321         2.838801     16.332623
up               0.565533         2.989578     22.512739


## 4. What this means in practice

The tested signals provide directional evidence that can help prioritize
content review. Better-performing pages generally show stronger search
and engagement metrics, while weaker pages may deserve closer review.

These signals should be used as decision-support rather than automatic
rules. A content reviewer should combine the signal with business
context before deciding whether a page should be refreshed.

In [8]:
print("Signal audit completed.")

Signal audit completed.
